# Views: document matching & fact review

Source/doc-type mapping and the fact-comparison review layer. Split out of
`notebook_01_document_matching` (Cell 7) and `notebook_03_fact_comparison` (Cells 9 and 11).

## Why views live in their own notebooks
`CREATE OR REPLACE VIEW` drops and recreates the view, which makes the **executing identity its
owner**. A view owned by someone else that reads it (e.g. `gold_person_evidence_summary`) then
fails with `INSUFFICIENT_PERMISSIONS`. So view DDL is kept out of the data-refresh notebooks:
views are *definitions*, not data, and only need (re)creating when a definition changes.

**Run this notebook only when a view definition changes**, not as part of a routine refresh.
Run it as the account that owns the views (or follow it with
`ALTER VIEW <name> OWNER TO <owner>` for each view it recreates).

## Run order
1. `v_source_doctype_map` (needs `silver_source`-style titles only; must exist before `gold_source_coverage` is populated).
2. `gold_fact_comparison_reviewed` -- prerequisite: `silver_fact_conflict_review` (`notebook_fact_conflict_review_init`) and the `gold_fact_comparison` table.
3. `gold_housekeeping_tree_gap_occupations` -- reads (2).


## Cell 7 — Source/doc_type mapping view

Maps silver_source title patterns to doc_type_detected vocabulary.
Extend the values list as new source_title values appear when processing additional surname branches.

In [0]:
%sql
CREATE OR REPLACE VIEW genealogy.v_source_doctype_map AS
SELECT source_title, doc_type_category FROM (VALUES
  -- England & Wales certificates
  ('Certified Copy of an Entry at Birth',              'BirthCertificate'),
  ('Certified Copy of an Entry of Marriage',           'MarriageCertificate'),
  ('Certified Copy of an Entry of Death',              'DeathCertificate'),
  ('Certified Copy of an entry at Death',              'DeathCertificate'),
  -- England census
  ('1841 England Census',                              'Census'),
  ('1851 England Census',                              'Census'),
  ('1861 England Census',                              'Census'),
  ('1871 England Census',                              'Census'),
  ('1881 England Census',                              'Census'),
  ('1891 England Census',                              'Census'),
  ('1901 England Census',                              'Census'),
  ('1911 England Census',                              'Census'),
  ('1921 England Census',                              'Census'),
  ('1939 England and Wales Register',                  'Census'),
  -- Scotland census
  ('1841 Scotland Census',                             'Census'),
  ('1851 Scotland Census',                             'Census'),
  ('1861 Scotland Census',                             'Census'),
  ('1871 Scotland Census',                             'Census'),
  ('1881 Scotland Census',                             'Census'),
  ('1891 Scotland Census',                             'Census'),
  ('1901 Scotland Census',                             'Census'),
  ('1911 Scotland Census',                             'Census'),
  ('1921 Scotland Census',                             'Census'),
  -- US census
  ('1850 United States Federal Census',                'Census'),
  ('1860 United States Federal Census',                'Census'),
  ('1870 United States Federal Census',                'Census'),
  ('1880 United States Federal Census',                'Census'),
  ('1890 United States Federal Census',                'Census'),
  ('1900 United States Federal Census',                'Census'),
  ('1910 United States Federal Census',                'Census'),
  ('1920 United States Federal Census',                'Census'),
  ('1930 United States Federal Census',                'Census'),
  ('1940 United States Federal Census',                'Census'),
  ('1950 United States Federal Census',                'Census'),
  -- Canada census
  ('1881 Census of Canada',                            'Census'),
  ('1891 Census of Canada',                            'Census'),
  ('1901 Census of Canada',                            'Census'),
  ('1911 Census of Canada',                            'Census'),
  ('1921 Census of Canada',                            'Census'),
  ('1931 Census of Canada',                            'Census'),
  -- Baptism / christening registers
  ('England, Select Births and Christenings, 1538-1975',                          'BirthIndex'),
  ('Scotland, Select Births and Baptisms, 1564-1950',                             'BirthIndex'),
  ('England & Wales Christening Records, 1530-1906',                              'BaptismRegister'),
  ('Nottinghamshire, England, Church of England Baptisms, 1813-1919',             'BaptismRegister'),
  ('Nottinghamshire, England, Church of England Baptisms, 1813-1922',             'BaptismRegister'),
  ('Cambridgeshire and Huntingdonshire, England, Church of England Births and Baptisms, 1813-1924', 'BaptismRegister'),
  ('Norfolk, England, Church of England Births and Baptisms, 1813-1919',          'BaptismRegister'),
  ('London, England, Births and Baptisms, 1813-1906',                             'BaptismRegister'),
  -- Marriage registers & OPR
  ('Nottinghamshire, England, Church of England Marriages and Banns, 1754-1937',  'MarriageCertificate'),
  ('Scotland OPR Banns & Marriages, 1553-1854',                                   'MarriageCertificate'),
  ('Cambridgeshire and Huntingdonshire, England, Church of England Marriages and Banns, 1754-1950', 'MarriageCertificate'),
  ('Norfolk, England, Church of England Marriages and Banns, 1754-1936',          'MarriageCertificate'),
  ('Derbyshire, England, Church of England Marriages and Banns, 1754-1932',       'MarriageCertificate'),
  ('London, England, Marriages and Banns, 1754-1921',                             'MarriageCertificate'),
  ('Sheffield and Rotherham, Yorkshire, England, Church of England Marriages and Banns, 1754-1948', 'MarriageCertificate'),
  ('Gretna Green, Scotland, Marriage Registers, 1794-1895', 'MarriageCertificate'),
  -- Death / burial registers
  ('Nottinghamshire, England, Church of England Burials, 1813-1994',               'BurialRegister'),
  ('Cambridgeshire and Huntingdonshire, England, Church of England Deaths and Burials, 1813-1997', 'BurialRegister'),
  ('Norfolk, England, Church of England Deaths and Burials, 1813-1995',            'BurialRegister'),
  ('Web: UK, Burial and Cremation Index, 1576-2014',                               'BurialRegister'),
  ('Scotland Statutory Death Register',                                            'DeathCertificate'),
  -- Birth indexes (replaces BMDIndex for birth-specific sources)
  ('England & Wales, Birth Index: 1984-2005',                                     'BirthIndex'),
  ('England & Wales, FreeBMD Birth Index, 1837-1915',                             'BirthIndex'),
  ('Scotland Statutory Register Births Index, 1855-',                             'BirthIndex'),
  -- Marriage indexes (replaces BMDIndex for marriage-specific sources)
  ('England & Wales, Marriage Index: 1916-2005',                                  'MarriageIndex'),
  ('England & Wales, Civil Registration Marriage Index, 1916-2005',               'MarriageIndex'),
  ('England & Wales, Civil Registration Marriage Index, 1837-1915',               'MarriageIndex'),
  ('England & Wales, FreeBMD Marriage Index: 1837-1915',                          'MarriageIndex'),
  ('Scotland Statutory Register Marriages Index, 1855-1930',                      'MarriageIndex'),
  ('England, Select Marriages, 1538-1973',                                        'MarriageIndex'),
  ('England, Select Marriages, 1538\u20131973',                                  'MarriageIndex'),
  ('Scotland, Select Marriages, 1561-1910',                                       'MarriageIndex'),
  -- Death indexes (replaces BMDIndex for death-specific sources)
  ('England & Wales, Death Index: 1984-2005',                                     'DeathIndex'),
  ('England and Wales, Death Index, 1989-2019',                                   'DeathIndex'),
  ('England & Wales, Civil Registration Death Index, 1916-2007',                  'DeathIndex'),
  ('England & Wales, FreeBMD Death Index: 1837-1915',                             'DeathIndex'),
  ('England, Select Deaths and Burials, 1538-1991',                               'DeathIndex'),
  -- Mixed BMD transcript (covers births, marriages AND burials in one source title —
  -- cannot be disambiguated by title alone, so mapped to OTHER)
  ('Norfolk, England, Transcripts of Church of England Baptism, Marriage and Burial Registers, 1600-1935', 'OTHER'),
  -- Index-style records with no document to hold -> NOT_APPLICABLE via OTHER_NO_FILE
  ('British Phone Books, 1880-1984', 'OTHER_NO_FILE'),
  ('UK and Ireland, Find a Grave Index, 1300s-Current', 'OTHER_NO_FILE'),
  ('England & Wales, National Probate Calendar (Index of Wills and Administrations), 1858-1995', 'OTHER_NO_FILE'),
  ('Scotland, National Probate Index (Calendar of Confirmations and Inventories), 1876-1936', 'OTHER_NO_FILE'),
  ('U.K., City and County Directories, 1600s-1900s', 'OTHER_NO_FILE'),
  ('1911 England Census Summary Books', 'OTHER_NO_FILE'),
  ('West Yorkshire, England, Electoral Registers, 1840-1962', 'OTHER_NO_FILE'),
  -- Missing siblings of existing entries
  ('Sheffield and Rotherham, Yorkshire, England, Church of England Baptisms, 1813-1923', 'BaptismRegister'),
  ('England & Wales, Civil Registration Birth Index, 1837-1915', 'BirthIndex'),
  -- Military records
  ('UK, World War I Service Records, 1914-1920',                                  'MilitaryRecord'),
  ('UK, British Army World War I Service Records, 1914-1920',                     'MilitaryRecord'),
  ('British Army WWI Service Records, 1914-1920',                                 'MilitaryRecord'),
  ('UK, World War II Service Records',                                            'MilitaryRecord'),
  ('UK, Soldiers Died in the Great War, 1914-1919',                               'MilitaryRecord'),
  ('UK,  British Army World War I Pension Records 1914-1920',                     'MilitaryRecord'),
  ('UK, Army Registers of Soldiers\' Effects, 1901-1929',                        'MilitaryRecord'),
  ('Australia, World War II Military Service Records, 1939-1945',                 'MilitaryRecord'),
  ('UK, Navy Lists, 1888-1970',                                                   'MilitaryRecord'),
  ('UK, Royal Air Force Airmen Records, 1918-1940',                               'MilitaryRecord'),
  -- Newspapers
  ('UK and Ireland, Newspapers.com\u2122 Marriage Index, 1800s-current',         'NewspaperClipping'),
  ('U.S., Newspapers.com Obituary Index, 1800s-current',                          'NewspaperClipping'),
  ('Canada, Newspapers.com Obituary Index, 1800s-current',                        'NewspaperClipping'),
  ('UK and Ireland, Newspapers.com\u2122 Obituary Index, 1800s-current',         'NewspaperClipping'),
  -- Electoral
  ('UK, Electoral Registers, 2003-2010',                                          'OTHER_NO_FILE'),
  ('UK, Electoral Registers, 2011-2018',                                          'OTHER_NO_FILE'),
  ('Australia, Electoral Rolls, 1903-1980',                                       'ElectoralRegister'),
  ('New Zealand, Electoral Rolls, 1853-2010',                                     'ElectoralRegister'),
  ('London, England, Electoral Registers, 1832-1965',                             'ElectoralRegister'),
  -- Passenger lists
  ('UK, Outward Passenger Lists, 1890-1960',                                      'PassengerList'),
  ('UK Incoming Passenger Lists, 1878-1960',                                      'PassengerList'),
  ('New York Passenger Lists, 1820-1957',                                         'PassengerList'),
  ('Pennsylvania, U.S., Arriving Passenger and Crew Lists, 1800-1962',            'PassengerList'),
  -- Other
  ('England & Wales, National Probate Calendar (Index of Wills and Administrations),1861-1941', 'Probate'),
  ('Scotland, Burial Monument Inscriptions, 1507-2019',                           'MonumentalInscription'),
  ('North Lanarkshire, Scotland, Poor Law Applications and Registers, 1849-1917', 'PoorLawRecord'),
  ('UK, City and County Directories, 1766 - 1946',                                'Directory'),
  ('England & Wales, Criminal Registers, 1791-1892',                              'CriminalRecord'),
  ('Web: Scotland, Prison Records Index, 1828-1878',                              'CriminalRecord'),
  ('UK, Calendar of Prisoners, 1868-1929',                                        'CriminalRecord')
) AS t(source_title, doc_type_category);

## Cell 9 — gold_fact_comparison_reviewed (applies review marks)

Prerequisite: `silver_fact_conflict_review` exists (notebook_fact_conflict_review_init).

Key: file_id + person_gedcom_id + fact_type + transcript_value (null-safe). The latest review
per key wins. `review_applies` is TRUE only while the tree value is unchanged since the
review (`reviewed_tree_value`); a changed tree value reopens the conflict, and a changed
transcript value or new file_id is simply a new key (open). Reviewed rows stay visible here.
`is_open_conflict` is the single definition of "conflict still needing attention".

`TREE_GAP` reviews (legitimate occupation missing from the tree) are the exception to the
tree-value snapshot rule: adding one occupation changes `tree_value`, which must not reopen the
person's other TREE_GAP rows as conflicts. They stay applied until the comparison is a MATCH.


In [0]:
%sql
CREATE OR REPLACE VIEW genealogy.gold_fact_comparison_reviewed AS
WITH latest_review AS (
  SELECT * EXCEPT (rn)
  FROM (
    SELECT r.*,
           ROW_NUMBER() OVER (
             PARTITION BY file_id, person_gedcom_id, fact_type, transcript_value
             ORDER BY reviewed_at DESC NULLS LAST
           ) AS rn
    FROM genealogy.silver_fact_conflict_review r
  )
  WHERE rn = 1
)
SELECT
  fc.*,
  rv.review_status,
  rv.reason_code        AS review_reason_code,
  rv.notes              AS review_notes,
  rv.reviewed_at,
  rv.reviewed_tree_value,
  COALESCE(rv.review_status IS NOT NULL
           AND (rv.reason_code = 'TREE_GAP' OR rv.reviewed_tree_value <=> fc.tree_value), FALSE)
                                                                              AS review_applies,
  (fc.status = 'CONFLICT'
   AND NOT COALESCE(rv.review_status IS NOT NULL
                    AND (rv.reason_code = 'TREE_GAP' OR rv.reviewed_tree_value <=> fc.tree_value), FALSE))
                                                                              AS is_open_conflict,
  -- Legitimate occupation the tree lacks: reviewed TREE_GAP and the comparison still isn't a MATCH.
  -- Clears on its own once the tree gains the occupation (status flips to MATCH on the next rebuild).
  COALESCE(rv.reason_code = 'TREE_GAP'
           AND fc.fact_type = 'occupation'
           AND fc.status IN ('CONFLICT', 'TRANSCRIPT_ONLY'), FALSE)           AS is_tree_gap
FROM genealogy.gold_fact_comparison fc
LEFT JOIN latest_review rv
  ON  rv.file_id          = fc.file_id
  AND rv.person_gedcom_id = fc.person_gedcom_id
  AND rv.fact_type        = fc.fact_type
  AND rv.transcript_value <=> fc.transcript_value;


## Cell 11 — gold_housekeeping_tree_gap_occupations (Housekeeping worklist)

One row per document occupation Ed has marked `TREE_GAP` that the tree still lacks, so he can add
a dated occupation in Ancestry. Rows vanish by themselves once the tree matches (no review step).
`NOT_AN_OCCUPATION` rows (students, club roles) are deliberately absent. Feeds
`SIGNAL_TRANSCRIPT_ONLY_FACTS` via `is_tree_gap`.


In [0]:
%sql
CREATE OR REPLACE VIEW genealogy.gold_housekeeping_tree_gap_occupations AS
SELECT
  fc.person_gedcom_id,
  fc.display_name,
  fc.file_id,
  fc.file_name,
  ot.year               AS document_year,
  fc.source_doc_type,
  fc.transcript_value   AS occupation_as_written,
  fc.tree_value         AS tree_occupations,
  fc.review_notes,
  fc.reviewed_at
FROM genealogy.gold_fact_comparison_reviewed fc
LEFT JOIN genealogy.ocr_transcriptions ot ON ot.file_id = fc.file_id
WHERE fc.is_tree_gap;
